In [6]:
import time
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd
import os
import requests
import io
from pathlib import Path
import random
from FinMind.data import DataLoader
from google.colab import userdata
import traceback
from collections import deque
# !pip install FinMind
# 投標前10日之:
# 大盤三大法人
# 大盤融資、券
# 美股漲幅-->四大指數
# 大盤漲幅、平均成交量
# 櫃買漲幅、平均成交量



In [7]:
# 取得df近10日資料
def get_near_n_day(df:pd.DataFrame, day=10, date_name="date"):

    df[date_name] = pd.to_datetime(df[date_name])
    unique_dates = sorted(df[date_name].unique())

    near_n_day = unique_dates[day-1:-1]
    near_n_day_df = df[df[date_name].isin(near_n_day)].copy()
    return near_n_day_df

# 取得台股大盤三大法人資訊
def get_market_Inst_tw(dl, target_time:pd.Timestamp, day=10):
    """
    外資 : Foreign_Investor
    投信 : Investment_Trust
    自營商: Dealer_Hedging + Dealer_self

    """
    end_date = target_time - timedelta(days=1)
    start_date = end_date - timedelta(days=30) # 抓長一點確保扣除假日有足夠天數
    df_total = dl.taiwan_stock_institutional_investors_total(start_date=start_date.strftime("%Y-%m-%d"),
                                                             end_date=end_date.strftime("%Y-%m-%d"))
    if isinstance(df_total, dict) and 'data' not in df_total:
        print(f"⚠️ API 回傳異常: {df_total.get('msg', '未知錯誤')}")
        return None
    near_n_day_df = get_near_n_day(df_total, day)
    near_n_day_df["name"] = near_n_day_df["name"].str.strip()

    near_n_day_df["net"] = near_n_day_df["buy"].astype(float) - near_n_day_df["sell"].astype(float)
    df_pivot = near_n_day_df.pivot_table(index="date", columns="name", values="net", aggfunc='sum')
    df_pivot["Dealer_total"] = df_pivot["Dealer_Hedging"] + df_pivot["Dealer_self"]

    avg_data = pd.Series({
        "外資平均增減": df_pivot['Foreign_Investor'].mean(),
        "投信平均增減": df_pivot['Investment_Trust'].mean(),
        "自營商平均增減": df_pivot['Dealer_total'].mean()
    })

    return avg_data

# 取得大盤融資、券資訊
def get_margin(dl, target_time:pd.Timestamp, day=10):
    """
    融資張數 : MarginPurchase
    融券張數 : ShortSale
    融資金額 : MarginPurchaseMoney
    """
    end_date = target_time - timedelta(days=1)
    start_date = end_date - timedelta(days=30) # 抓長一點確保扣除假日有足夠天數
    df_total = dl.taiwan_stock_margin_purchase_short_sale_total(start_date=start_date.strftime("%Y-%m-%d"),
                                                                end_date=end_date.strftime("%Y-%m-%d"))
    near_n_day_df = get_near_n_day(df_total, day)
    near_n_day_df["name"] = near_n_day_df["name"].str.strip()

    near_n_day_df["change"] = near_n_day_df["TodayBalance"] - near_n_day_df["YesBalance"]
    df_pivot = near_n_day_df.pivot_table(index="date", columns="name", values="change")

    avg_data = pd.Series({
        "融資張數增減": df_pivot['MarginPurchase'].mean().round(3),
        "融券張數增減": df_pivot['ShortSale'].mean().round(3),
        "融資金額增減": df_pivot['MarginPurchaseMoney'].mean().round(3)
    })
    return avg_data


dl = DataLoader()
token = userdata.get('finmind')
dl.login_by_token(api_token=token)
re = get_market_Inst_tw(dl, pd.to_datetime("2024-08-21"))
re



2026-02-21 13:15:12.185 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-21 13:15:12.830 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-21 13:15:12.843 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 


,0
外資平均增減,7.769766e+09
投信平均增減,6.176848e+09
自營商平均增減,-7.007950e+09


In [8]:
def get_market_usa(target_time:pd.Timestamp, day=10):
    """
    ex.
    ^DJI    -0.750
    ^GSPC    0.353
    ^IXIC    0.659
    ^SOX     6.164
    """
    # 四大指數代碼
    tickers = ["^GSPC", "^IXIC", "^DJI", "^SOX"]
    name_map = {
        "^GSPC": "標普500_10日漲幅(%)",
        "^IXIC": "那斯達克_10日漲幅(%)",
        "^DJI": "道瓊工業_10日漲幅(%)",
        "^SOX": "費城半導體_10日漲幅(%)"
    }
    end_date = target_time - timedelta(days=1)
    start_date = end_date - timedelta(days=30) # 抓長一點確保扣除假日有足夠天數
    df_total = yf.download(tickers, start=start_date, end=end_date)

    close_df = df_total["Close"].copy()
    close_df.reset_index(inplace=True)
    near_n_day_df = get_near_n_day(close_df, 10, "Date")
    diff = ((near_n_day_df.iloc[-1, 1:] - near_n_day_df.iloc[0, 1:]) / near_n_day_df.iloc[0, 1:]) * 100  # 單位:百分比
    diff = diff.astype(float).round(3)
    diff = diff.rename(index=name_map)
    return diff

# re = get_market_usa(pd.to_datetime("2024-07-31"))
# print(re)

In [9]:
# 1. 讀取當前要抓取的類型的儲存表沒有則創建
# 2. 產生與競拍資訊標中新增的股票代號跟投標開始日
def search_index_list(raw_data_path, curr_data_path, code_col:str, time_col:str, feature_cols:list):
    """
    curr_data: 上市櫃行情表, pd.DataFrame
    diff_index: 與原始競拍資料差異, pd.Index
    """
    if not os.path.exists(raw_data_path):
        return
    raw_data = pd.read_csv(raw_data_path, encoding="utf-8-sig", dtype={code_col:str})
    raw_data[time_col] = pd.to_datetime(raw_data[time_col], format="mixed")
    if os.path.exists(curr_data_path):
        curr_data = pd.read_csv(curr_data_path, encoding="utf-8-sig", dtype={code_col:str})
        curr_data[time_col] = pd.to_datetime(curr_data[time_col], format="mixed")
    else:
        curr_data = pd.DataFrame(columns=[code_col, time_col]+feature_cols)

    raw_indexed = raw_data.set_index([code_col, time_col])

    if not curr_data.empty:
        curr_indexed = curr_data.set_index([code_col, time_col])
        diff_index = raw_indexed.index.difference(curr_indexed.index)
    else:
        diff_index = raw_indexed.index

    return curr_data, diff_index

# save_folder = Path("/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project/csv")
# raw_data_path = save_folder / "bid_info.csv"
# market_stmt_path = save_folder / "all_market_info.csv"

# a = search_index_list(raw_data_path, market_stmt_path, "證券代號", "投標開始日")
# a


In [10]:
# 取得近十日大盤、櫃買平均漲跌幅與成交量
def get_market_tw(target_time:pd.Timestamp, days=10):
    """
        {'大盤_平均漲跌幅(%)': np.float64(3.2869),
        '大盤_平均成交量': np.float64(5958110.0),
        '櫃買_平均漲跌幅(%)': np.float64(-2.5435),
        '櫃買_平均成交量': np.float64(949820.0)}
    """
    end_date = target_time
    start_date = end_date - timedelta(days=40) # 抓長一點確保扣除假日有足夠天數
    re = {}
    for code in ["^TWII", "^TWOII"]:
        df = yf.download(code,
                        start=start_date.strftime('%Y-%m-%d'),
                        end=end_date.strftime('%Y-%m-%d'),
                        auto_adjust=False, # 關閉自動校正，確保看到原本的 Close
                        progress=False)
        if df.empty:
            return f"找不到 {code} 的數據"

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        price_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'

        final_df = df.tail(days).copy()

        if len(final_df) < days:
            return f"數據不足：要求 {days} 天，但僅抓到 {len(final_df)} 天"

        price_start = final_df[price_col].iloc[0]
        price_end = final_df[price_col].iloc[-1]

        total_change_pct = ((price_end - price_start) / price_start) * 100
        avg_volume = final_df['Volume'].mean()

        code_name = "大盤" if code == "^TWII" else "櫃買"
        re[f"{code_name}_10日漲幅(%)"] = round(total_change_pct, 3)
        re[f"{code_name}_平均成交量"] = round(avg_volume, 0)
        re = pd.Series(re)

    return re

# # --- 測試 ---
# # code 使用 '^TWII' (大盤) 與 '^TWOII' (櫃買)
# date = pd.to_datetime("2024-07-31")
# display(get_market_tw(date))
# display(get_market_indices("^TWOII", "2026-02-12"))

In [11]:
def sleep_until_next_hour():
    now = datetime.now()
    # 睡到下個整點的 05 秒
    next_hour = (now + timedelta(hours=1)).replace(minute=0, second=5, microsecond=0)
    wait_sec = (next_hour - now).total_seconds()

    # 防止負數或過短的等待
    if wait_sec > 0:
        time.sleep(wait_sec)
    print(f"進入新時段:{datetime.now().hour} 點，恢復抓取！")

In [12]:
feature_cols = [
    "外資平均增減", "投信平均增減", "自營商平均增減",
    "融資張數增減", "融券張數增減", "融資金額增減",
    "道瓊工業_10日漲幅(%)", "標普500_10日漲幅(%)", "那斯達克_10日漲幅(%)", "費城半導體_10日漲幅(%)",
    "大盤_10日漲幅(%)", "大盤_平均成交量", "櫃買_10日漲幅(%)", "櫃買_平均成交量"
]

def main():
    save_folder = Path("/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project/csv")
    raw_data_path = save_folder / "bid_info.csv"
    market_stmt_path = save_folder / "all_market_info.csv"
    start = time.time()
    curr_data, diff_index = search_index_list(raw_data_path, market_stmt_path, "證券代號", "投標開始日", feature_cols)

    if diff_index.empty:
        print("沒有新增之競拍股票")
        return
    dl = DataLoader()
    token = userdata.get('finmind')
    dl.login_by_token(api_token=token)

    last_hour = datetime.now().hour
    processed_count = 0
    max_safe_limit = 300 # 設定一小時最多跑 300 次 (600的一半，因為每次迴圈會消耗2個token)
    track = deque()
    for idx in diff_index:
        current_hour = datetime.now().hour
        if current_hour != last_hour:
            print(f"API 次數已重置")
            processed_count = 0  # 重置次數
            last_hour = current_hour # 更新紀錄點

        if processed_count >= max_safe_limit:
            print(f"本小時已達 {max_safe_limit} 次上限，休息中...")
            sleep_until_next_hour()
            # 睡醒後，一定要手動更新一次 last_hour，否則會重複觸發 sleep
            last_hour = datetime.now().hour
            processed_count = 0
        try:
            code = idx[0]
            target_time = idx[1]
            print(f"股票代號 : {code} | 開標日期 : {target_time}")
            tw_indices = get_market_Inst_tw(dl, target_time, 10) # 三大法人
            tw_margin = get_margin(dl, target_time, 10)          # 融資融券
            usa_market = get_market_usa(target_time, 10)         # 美股四大指數漲幅
            tw_market = get_market_tw(target_time, 10)           # 大盤櫃買漲幅、平均成交量
            key_info = pd.Series({"證券代號": code,"投標開始日": target_time})
            combined_results = pd.concat([key_info, tw_indices, tw_margin, usa_market, tw_market])
            new_entry = pd.DataFrame([combined_results])
            curr_data = pd.concat([curr_data, new_entry], ignore_index=True)
            processed_count += 1
            time.sleep(random.uniform(1, 3))

        except Exception as e:
            print(f"{code}開標時間{target_time}發生錯誤: {e}")
            track.append((code, target_time, 0))
            traceback.print_exc()

    while track:
        code, target_time, try_count = track.popleft()
        try:
            code, target_time, try_count = track.popleft() # <--- 這是 deque 專用的高效方法
            print(f"處理中 (第 {try_count} 次): {code} | 開標日期 : {target_time}")

            tw_indices = get_market_Inst_tw(dl, target_time, 10) # 三大法人
            tw_margin = get_margin(dl, target_time, 10)          # 融資融券
            usa_market = get_market_usa(target_time, 10)         # 美股四大指數漲幅
            tw_market = get_market_tw(target_time, 10)           # 大盤櫃買漲幅、平均成交量

            key_info = pd.Series({"證券代號": code,"投標開始日": target_time})
            combined_results = pd.concat([key_info, tw_indices, tw_margin, usa_market, tw_market])
            new_entry = pd.DataFrame([combined_results])
            curr_data = pd.concat([curr_data, new_entry], ignore_index=True)
            time.sleep(random.uniform(1, 3))
        except Exception as e:
            if try_count < 3:
                track.append((code, target_time, try_count + 1))
                print(f"{code} 失敗，排隊等第 {try_count + 1} 次嘗試")
            else:
                # 試過 3 次了，徹底放棄
                print(f"{code} 失敗達 3 次，放棄。")


    curr_data.to_csv(market_stmt_path, index=False, encoding='utf-8-sig') # 如果是存 CSV
    print(f"成功更新 {len(diff_index)} 筆競拍資料！")
    end = time.time()
    print(f"耗時: {end - start} 秒")

if __name__ == "__main__":
    main()


2026-02-21 13:15:15.331 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-21 13:15:15.940 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-21 13:15:15.944 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 


股票代號 : 1294 | 開標日期 : 2024-09-05 00:00:00


2026-02-21 13:15:16.440 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalMarginPurchaseShortSale, data_id: 
/tmp/ipython-input-3153936204.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_total = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  4 of 4 completed
2026-02-21 13:15:20.137 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 
2026-02-21 13:15:20.320 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalMarginPurchaseShortSale, data_id: 


股票代號 : 1295 | 開標日期 : 2025-05-12 00:00:00


/tmp/ipython-input-3153936204.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_total = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  4 of 4 completed
2026-02-21 13:15:23.862 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 
2026-02-21 13:15:24.045 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalMarginPurchaseShortSale, data_id: 


...

股票代號 : 8272 | 開標日期 : 2024-10-02 00:00:00


/tmp/ipython-input-3153936204.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_total = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  4 of 4 completed
2026-02-21 13:21:38.174 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 
2026-02-21 13:21:38.356 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTotalMarginPurchaseShortSale, data_id: 


股票代號 : 8487 | 開標日期 : 2024-03-11 00:00:00


/tmp/ipython-input-3153936204.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_total = yf.download(tickers, start=start_date, end=end_date)
[*********************100%***********************]  4 of 4 completed


成功更新 132 筆競拍資料！
耗時: 387.2371039390564 秒
